In [46]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import os
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [47]:
DATA_DIR = "/kaggle/input/datasets/andrewmvd/medical-mnist"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

Device: cuda


In [48]:
train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [49]:
#full_ds_train = datasets.ImageFolder(DATA_DIR, transform=train_tfms)
#full_ds_val = datasets.ImageFolder(DATA_DIR, transform=val_tfms)

#val_frac = 0.2
#val_size = int(len(full_ds_train) * val_frac)
#train_size = len(full_ds_train) - val_size

#generator = torch.Generator().manual_seed(42)

#train_ds, val_indices = random_split(
#    full_ds_train,
#    [train_size, val_size],
#    generator=generator
#)

#vals_ds = torch.utils.data.Subset(full_ds_val, val_indices.indices)

full_ds = datasets.ImageFolder(DATA_DIR, transform=train_tfms)
class_names = full_ds.classes
num_classes = len(class_names)

print("Classes:", class_names)
print("Total images:", len(full_ds))

val_frac = 0.2
val_size = int(len(full_ds) * val_frac)
train_size = len(full_ds) - val_size

train_ds, val_ds = random_split(
    full_ds,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(seed)
)

# IMPORTANT: turn off augmentations for validation
val_ds.dataset.transform = val_tfms

Classes: ['AbdomenCT', 'BreastMRI', 'CXR', 'ChestCT', 'Hand', 'HeadCT']
Total images: 58954


In [50]:
BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)

In [51]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

In [52]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

In [55]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader: 
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return running_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)
        running_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)

        correct += (preds == y).sum().item()
        total += y.size(0)

        all_preds.append(preds.cpu().numpy())
        all_labels.append(y.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    return running_loss / total, correct / total, all_labels, all_preds

In [56]:
EPOCHS = 8
best_val_acc = 0.0

train_losses, train_accs = [], []
val_losses, val_accs = [], []

for epoch in range(EPOCHS):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    va_loss, va_acc, y_true, y_pred = evaluate(model, val_loader, criterion)

    scheduler.step()

    train_losses.append(tr_loss)
    train_accs.append(tr_acc)
    val_losses.append(va_loss)
    val_accs.append(va_acc)

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
          f"val loss {va_loss:.4f} acc {va_acc:.4f}")

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), "best_resnet18_medmnist.pth")
        print("Saved best model!")

print("Best validation accuracy:", best_val_acc)

Epoch 1/8 | train loss 0.0049 acc 0.9989 | val loss 0.0040 acc 0.9989
Saved best model!
Epoch 2/8 | train loss 0.0010 acc 0.9998 | val loss 0.0000 acc 1.0000
Saved best model!
Epoch 3/8 | train loss 0.0000 acc 1.0000 | val loss 0.0000 acc 1.0000
Epoch 4/8 | train loss 0.0000 acc 1.0000 | val loss 0.0000 acc 1.0000
Epoch 5/8 | train loss 0.0000 acc 1.0000 | val loss 0.0000 acc 1.0000
Epoch 6/8 | train loss 0.0000 acc 1.0000 | val loss 0.0000 acc 1.0000
Epoch 7/8 | train loss 0.0000 acc 1.0000 | val loss 0.0000 acc 1.0000
Epoch 8/8 | train loss 0.0000 acc 1.0000 | val loss 0.0000 acc 1.0000
Best validation accuracy: 1.0


In [57]:
model.load_state_dict(torch.load("best_resnet18_medmnist.pth", map_location=device))

va_loss, va_acc, y_true, y_pred = evaluate(model, val_loader, criterion)
print("Final (best) val accuracy:", va_acc)

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

Final (best) val accuracy: 1.0
Confusion Matrix:
 [[1972    0    0    0    0    0]
 [   0 1793    0    0    0    0]
 [   0    0 1976    0    0    0]
 [   0    0    0 2029    0    0]
 [   0    0    0    0 1973    0]
 [   0    0    0    0    0 2047]]

Classification Report:
              precision    recall  f1-score   support

   AbdomenCT       1.00      1.00      1.00      1972
   BreastMRI       1.00      1.00      1.00      1793
         CXR       1.00      1.00      1.00      1976
     ChestCT       1.00      1.00      1.00      2029
        Hand       1.00      1.00      1.00      1973
      HeadCT       1.00      1.00      1.00      2047

    accuracy                           1.00     11790
   macro avg       1.00      1.00      1.00     11790
weighted avg       1.00      1.00      1.00     11790



In [58]:
va_loss, va_acc, y_true, y_pred = evaluate(model, val_loader, criterion)
print("Final (best) val accuracy:", va_acc)

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

Final (best) val accuracy: 1.0
Confusion Matrix:
 [[1972    0    0    0    0    0]
 [   0 1793    0    0    0    0]
 [   0    0 1976    0    0    0]
 [   0    0    0 2029    0    0]
 [   0    0    0    0 1973    0]
 [   0    0    0    0    0 2047]]

Classification Report:
              precision    recall  f1-score   support

   AbdomenCT       1.00      1.00      1.00      1972
   BreastMRI       1.00      1.00      1.00      1793
         CXR       1.00      1.00      1.00      1976
     ChestCT       1.00      1.00      1.00      2029
        Hand       1.00      1.00      1.00      1973
      HeadCT       1.00      1.00      1.00      2047

    accuracy                           1.00     11790
   macro avg       1.00      1.00      1.00     11790
weighted avg       1.00      1.00      1.00     11790

